# Managed Table vs External Table in Databricks

Databricks supports two types of tables:

1. **Managed Tables**
2. **External Tables**

The main difference lies in **who manages the data files and their lifecycle**.

---

# Managed Table

A **Managed Table** is a table where Databricks/Unity Catalog manages both:

- Table metadata
- Underlying data files

When a managed table is created, Databricks stores the data in its managed storage location.

### Example

```python
df.write.saveAsTable("movies")
```

or

```sql
CREATE TABLE movies (
    id INT,
    title STRING
);
```

---

## Managed Table Architecture

```text
Table
   ↓
Metadata Managed by Databricks
   ↓
Data Files Managed by Databricks
```

---

## What Happens When You Drop a Managed Table?

```sql
DROP TABLE movies;
```

Result:

```text
✅ Metadata Deleted
✅ Data Files Deleted
```

The table and its data are completely removed.

---

## Data Governance Benefits

Since Databricks controls both metadata and data:

✅ Easier governance

✅ Easier lifecycle management

✅ Centralized security

✅ Better compliance management

✅ No orphaned files

✅ Automatic cleanup

---

## Common Use Cases

Managed tables are best for:

```text
Internal Analytics
Temporary Tables
Intermediate ETL Outputs
Machine Learning Feature Tables
Sandbox Environments
```

Use managed tables when:

> Databricks owns the complete data lifecycle.

---

# External Table

An **External Table** is a table where Databricks manages only the metadata.

The actual data remains in a user-defined location such as:

```text
S3
ADLS
GCS
Volumes
External Locations
Data Lakes
```

### Example

```python
df.write \
  .option("path", "/Volumes/workspace/default/movies_data") \
  .saveAsTable("movies_ext")
```

or

```sql
CREATE TABLE movies_ext
USING PARQUET
LOCATION '/Volumes/workspace/default/movies_data';
```

---

## External Table Architecture

```text
Metadata
     ↓
Managed by Databricks

Data Files
     ↓
Managed by User
```

---

## What Happens When You Drop an External Table?

```sql
DROP TABLE movies_ext;
```

Result:

```text
✅ Metadata Deleted
❌ Data Files Remain
```

Only the table definition is removed.

The actual files remain available in the storage location.

---

## Data Governance Benefits

External tables are ideal for enterprise data lakes.

✅ Data can be shared across multiple platforms

✅ Data survives table deletion

✅ Supports centralized data lake architecture

✅ Multiple tools can access the same data

Example:

```text
S3 Data Lake
      ↓
Databricks
Snowflake
Athena
EMR
Power BI
```

All tools access the same underlying data.

---

## Governance Challenges

⚠️ Data lifecycle must be managed separately

⚠️ Risk of orphan files

⚠️ Storage permissions require careful management

⚠️ More governance responsibility on data teams

---

# Key Differences

| Feature | Managed Table | External Table |
|----------|----------|----------|
| Metadata Managed By | Databricks | Databricks |
| Data Managed By | Databricks | User |
| Storage Location | Managed Storage | User-Specified Storage |
| Data Ownership | Databricks | User |
| DROP TABLE Behavior | Deletes Metadata + Data | Deletes Metadata Only |
| Governance Complexity | Simple | More Advanced |
| Cross-Platform Access | Limited | Excellent |
| Best For | Internal Analytics | Enterprise Data Lakes |

---

# When Should You Use Managed Tables?

Use Managed Tables when:

```text
✅ Databricks owns the complete data lifecycle
✅ Temporary or intermediate data
✅ Internal analytics workloads
✅ Simpler governance is preferred
```

---

# When Should You Use External Tables?

Use External Tables when:

```text
✅ Data already exists in S3/ADLS/GCS
✅ Multiple platforms need access
✅ Enterprise data lake architecture
✅ Data should survive table deletion
✅ Long-term governed storage
```

---

# Interview Answer

> A Managed Table is a table in which Databricks manages both the metadata and the underlying data files. When the table is dropped, both the metadata and data are deleted. Managed tables provide simpler governance, centralized security, and automatic lifecycle management, making them suitable for internal analytics and temporary datasets.
>
> An External Table is a table where Databricks manages only the metadata, while the data remains in a user-defined storage location such as S3, ADLS, GCS, or Volumes. Dropping an external table removes only the metadata and leaves the data intact. External tables are preferred in enterprise data lake architectures where data must be shared across multiple tools while maintaining centralized governance.

# How to Check Whether a Table is Managed or External in Databricks

## Method 1: DESCRIBE EXTENDED

Run:

```sql
DESCRIBE EXTENDED table_name;
```

Example:

```sql
DESCRIBE EXTENDED movies;
```

Look for:

```text
Type: MANAGED
```

or

```text
Type: EXTERNAL
```

This is the easiest and most commonly used method.

---

## Method 2: DESCRIBE DETAIL

Run:

```sql
DESCRIBE DETAIL table_name;
```

Example:

```sql
DESCRIBE DETAIL movies;
```

Output contains a field similar to:

```text
type = MANAGED
```

or

```text
type = EXTERNAL
```

You will also see:

```text
location
sizeInBytes
numFiles
format
```

and other useful metadata.

---

## Method 3: Using Spark SQL

```python
spark.sql("DESCRIBE DETAIL movies").show(truncate=False)
```

Example output:

```text
+--------+---------+
|name    |movies   |
|type    |MANAGED  |
|format  |delta    |
+--------+---------+
```

---

## Method 4: Check Table Location

Run:

```sql
DESCRIBE DETAIL movies;
```

Look at the location.

### Managed Table

Location is usually under Databricks-managed storage:

```text
dbfs:/user/hive/warehouse/movies
```

or Unity Catalog managed storage location.

### External Table

Location points to a user-specified path:

```text
/Volumes/workspace/default/movies_data

s3://bucket/movies

abfss://container@storage.dfs.core.windows.net/movies
```

---

## Interview Answer

> To determine whether a table is Managed or External in Databricks, use `DESCRIBE EXTENDED table_name` or `DESCRIBE DETAIL table_name`. These commands display the table metadata, including the table type. If the type is `MANAGED`, Databricks manages both metadata and data files. If the type is `EXTERNAL`, Databricks manages only the metadata while the data remains in a user-defined storage location.



In [0]:
# To see the metadata or we can go to catalog and go to details 